# Step 4: Final Optimized Cross-Task Evaluation

This version combines the best approaches:
1. **Architectural Integrity**: Uses explicit model loading with `bicubic` interpolation for positional embeddings
2. **Execution Speed**: Retains the O(1) GPU-vectorized lookup table for COCO-to-Cityscapes class mapping.
3. **Dataset Handling**: Uses the existing `LightningDataModule` for clean data loading.

In [1]:
# @title Install Dependencies
!pip install lightning > /dev/null
!pip install gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null

In [2]:
import os
import sys
import json
import yaml
import torch
import importlib
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Environment Configuration
from google.colab import drive
drive.mount('/content/drive')
project_root = '/content/drive/MyDrive/FundGitHubProject'
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder)

from eval.iouEval import iouEval
from eomt.models.vit import ViT
from eomt.models.eomt import EoMT
from eomt.training.mask_classification_semantic import MaskClassificationSemantic
from eomt.training.mask_classification_panoptic import MaskClassificationPanoptic

seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cuda


In [3]:

IMG_SIZE = (640, 640)

def _load_state_dict_into(model, ckpt_path):
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    if not any(k.startswith('network.') for k in state):
        state = {f'network.{k}': v for k, v in state.items()}

    model_sd = model.state_dict()
    for key in list(state.keys()):
        if 'pos_embed' not in key:
            continue
        if key not in model_sd:
            continue
        ckpt_shape  = state[key].shape
        model_shape = model_sd[key].shape
        if ckpt_shape == model_shape:
            continue

        N_ckpt, D   = ckpt_shape[1], ckpt_shape[2]
        N_model     = model_shape[1]
        H_c = W_c   = int(N_ckpt  ** 0.5)
        H_m = W_m   = int(N_model ** 0.5)

        pe = state[key]
        pe = pe.reshape(1, H_c, W_c, D).permute(0, 3, 1, 2)
        pe = F.interpolate(pe.float(), size=(H_m, W_m),
                           mode='bicubic', align_corners=False)
        pe = pe.permute(0, 2, 3, 1).reshape(1, N_model, D)
        state[key] = pe
        print(f'  interpolated {key}: {list(ckpt_shape)} -> {list(model_shape)}')

    model.load_state_dict(state, strict=False)
    print(f'  loaded {ckpt_path}')
    return model

def load_cs_model(ckpt_path):
    encoder = ViT(img_size=IMG_SIZE, backbone_name='vit_base_patch14_reg4_dinov2')
    network = EoMT(encoder, num_classes=19, num_q=100, num_blocks=3)
    model = MaskClassificationSemantic(
        network=network, img_size=IMG_SIZE, num_classes=19, attn_mask_annealing_enabled=False
    )
    return _load_state_dict_into(model, ckpt_path).eval().to(device)

def load_coco_model(ckpt_path):
    stuff_classes = list(range(80, 133))
    encoder = ViT(img_size=IMG_SIZE, backbone_name='vit_base_patch14_reg4_dinov2')
    network = EoMT(encoder, num_classes=133, num_q=200, num_blocks=3)
    model = MaskClassificationPanoptic(
        network=network, img_size=IMG_SIZE, num_classes=133,
        stuff_classes=stuff_classes, attn_mask_annealing_enabled=False
    )
    return _load_state_dict_into(model, ckpt_path).eval().to(device)

In [4]:
## 3. Instantiate Models and DataModule
from eomt.datasets.cityscapes_semantic import CityscapesSemantic

print('Loading EoMT-Cityscapes...')
model_cs = load_cs_model("eomt/eomt_weights/eomt_cityscapes.bin")

print('\nLoading EoMT-COCO...')
model_coco = load_coco_model("eomt/eomt_weights/eomt_coco.bin")

dm_cs = CityscapesSemantic(path="eomt/data", batch_size=1, num_workers=2, img_size=IMG_SIZE)
dm_cs.setup("validate")

Loading EoMT-Cityscapes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


  interpolated network.encoder.backbone.pos_embed: [1, 4096, 768] -> [1, 1600, 768]
  loaded eomt/eomt_weights/eomt_cityscapes.bin

Loading EoMT-COCO...
  loaded eomt/eomt_weights/eomt_coco.bin


In [5]:
## 4 COCO Zero-Shot Evaluation
mapping_path = os.path.join(project_root, 'coco-classes-mapping-master/coco_to_cs.json')
with open(mapping_path, 'r') as f:
    numerical_map = json.load(f)

lookup_table = torch.full((256,), 19, dtype=torch.long, device=device)
for model_idx_str, cs_id in numerical_map.items():
    lookup_table[int(model_idx_str)] = cs_id

evaluator = iouEval(20)

for batch in tqdm(dm_cs.val_dataloader(), desc="Evaluating Mapped COCO"):
    imgs, targets = batch
    gt = model_cs.to_per_pixel_targets_semantic(targets, 19)[0].to(device)

    with torch.no_grad():
        tx = model_coco.resize_and_pad_imgs_instance_panoptic([imgs[0].to(device)])
        mp, cp = model_coco(tx)
        mp = model_coco.revert_resize_and_pad_logits_instance_panoptic(
            F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"),
            [imgs[0].shape[-2:]]
        )
        pred = model_coco.to_per_pixel_preds_panoptic(
            mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8
        )[0][..., 0]

        mapped_pred = lookup_table[pred.long()]
        evaluator.addBatch(mapped_pred.unsqueeze(0).unsqueeze(0), gt.unsqueeze(0).unsqueeze(0))

_, ious = evaluator.getIoU()
print(f"\nUpdated Zero-Shot COCO mIoU on Cityscapes: {ious[:19].mean()*100:.2f}%")

Evaluating Mapped COCO: 100%|██████████| 500/500 [04:07<00:00,  2.02it/s]


Updated Zero-Shot COCO mIoU on Cityscapes: 48.82%


In [6]:
## 5. Supervised Evaluation (Cityscapes)
## we have to fix the sliding window from 640 to 1024


evaluator_cs = iouEval(20)

for batch in tqdm(dm_cs.val_dataloader(), desc="Evaluating Cityscapes Model"):
    imgs, targets = batch
    gt = model_cs.to_per_pixel_targets_semantic(targets, 19)[0].to(device)

    with torch.no_grad():
        crops, origins = model_cs.window_imgs_semantic([imgs[0].to(device)])
        m_l, c_l = model_cs(crops)
        m_l = F.interpolate(m_l[-1], model_cs.img_size, mode="bilinear")
        crop_logits = model_cs.to_per_pixel_logits_semantic(m_l, c_l[-1])
        pred_cs = model_cs.revert_window_logits_semantic(crop_logits, origins, [imgs[0].shape[-2:]])[0].argmax(0)

        evaluator_cs.addBatch(pred_cs.unsqueeze(0).unsqueeze(0), gt.unsqueeze(0).unsqueeze(0))

_, ious_cs = evaluator_cs.getIoU()
print(f"\nCityscapes Supervised mIoU: {ious_cs[:19].mean()*100:.2f}%")

Evaluating Cityscapes Model: 100%|██████████| 500/500 [04:56<00:00,  1.69it/s]


Cityscapes Supervised mIoU: 79.73%
